# 最適執行ビジュアルラボ

## エグゼクティブサマリー
執行を速めるほど市場インパクトは増え、遅くするほど価格変動（タイミング）リスクが増えます。本ノートブックは、この基本的なトレードオフを、単一の再現可能な合成市場のうえで一貫して扱います。Almgren–Chriss のスケジューリング、板の回復力（resilient liquidity）、反応型指値注文板、取引コスト分析（TCA）、制約付き強化学習による調整を、共通乱数で対にして比較します。強化学習の整形報酬と経済的な実装ショートフォールは、常に明確に分離します。

*本ページは日本語版です（英語版: `01_optimal_execution_visual_lab.html`）。*

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from optimal_execution.config import load_config
from optimal_execution.provenance import artifact_dirs

config_path = Path(os.environ.get("OPTIMAL_EXECUTION_CONFIG", "configs/quick.yaml"))
cfg = load_config(config_path)
paths = artifact_dirs(cfg)
display(
    Markdown(
        f"**プロファイル:** `{cfg.profile}` · **シード:** `{cfg.seed}` · **注文:** `{cfg.initial_inventory:,.0f}` 株 · **時間軸:** `{cfg.horizon_seconds:.0f}` 秒"
    )
)

## コンセプトマップと市場マイクロストラクチャーの定義

**スプレッド**は最良気配をまたいで即時に約定する費用、**流動性**は表示深度と補充力の総合、**マーケットインパクト**は自分の取引が価格に残す足跡、**タイミングリスク**は不動（unaffected）価格が動くなかで在庫を持ち続けることの不確実性です。**執行方策**は、時刻・在庫・板の状態を、上限付きの成行・指値注文へ写像します。

戦略的ベースライン（AC ／ 回復力スケジュール）→ 戦術的ヒューリスティックまたは RL 調整 → 安全層（在庫・参加率・価格カラー・期限）。

In [ ]:
display(
    pd.DataFrame(
        {
            "variable": [
                "side",
                "arrival price",
                "initial inventory",
                "steps",
                "annualized volatility",
                "ADV",
                "spread",
                "participation cap",
            ],
            "value": [
                cfg.side,
                cfg.arrival_price,
                cfg.initial_inventory,
                cfg.n_decision_steps,
                cfg.annualized_volatility,
                cfg.average_daily_volume,
                cfg.spread_bps,
                cfg.max_participation_rate,
            ],
            "unit": [
                "",
                "price/share",
                "shares",
                "",
                "fraction/year",
                "shares/day",
                "bps",
                "fraction",
            ],
        }
    )
)

## 合成市場プロファイルと不動価格パス
出来高・ボラティリティ・スプレッド・深度は、局所的な確率乗数を伴う日中プロファイルに従います。不動（unaffected）価格には自己インパクトが一切含まれず、被影響価格はその上に重ねられます。共通乱数により、戦略間の差は独立ではなく対（ペア）として測られます。

In [ ]:
display(Image(filename=str(paths["figures"] / "unaffected_price_paths.png")))

## 一時的・恒久的・過渡的インパクトと平方根則

3 つの線形経路を区別します。**一時的（temporary）**インパクトは取引中だけ支払う価格譲歩で、1 株あたり $g(v_k)=\eta\,v_k$（レート $v_k=q_k/\Delta t$）、1 ステップの現金コストは $C^{\text{temp}}_k=\eta\,q_k^2/\Delta t$。取引が終われば痕跡を残しません。**恒久的（permanent）**インパクトは減衰しない仲値シフト $\gamma\sum_{j<k}q_j$ で、後続の取引が引き継ぐ足跡です。**過渡的（transient）**インパクトは、取引で積み上がり回復率 $\rho$ で緩和する変位状態 $D$ で、点インパルス規約を用います。

$$D_{k+1}=e^{-\rho\,\Delta t}\left(D_k+\eta_t q_k\right).$$

ステップ $k$ の約定は取引前の $D_k$ に自分のジャンプの半分を加えた $D_k+\tfrac{1}{2}\eta_t q_k$（板を通した区間平均）を負担します。符号規約：各量は正の不利量で、売りは $S^0-a$、買いは $S^0+a$ で約定します。

指数則は伝播核（propagator）の $\rho$ 特殊ケースです。

$$D_k=\sum_{j<k} G\!\left((k-j)\Delta t\right)\eta_t q_j,\qquad G(0)=1,$$

指数核または完全単調な冪核に対して成り立ちます。シミュレータがこれらの潜在状態を公開しているため、各取引の一時的／恒久的／過渡的への分解はモデル内で厳密になります——実データでは決して得られない特権です。

平方根則 $I(Q)=Y\,\sigma_{\text{day}}\sqrt{Q/\mathrm{ADV}}$ は、サイズ $Q$ のメタオーダーの**総**インパクトに対する凹関数で、経験的診断としてのみ示し、線形経路の上には加算しません（二重計上を避けるため）。

In [ ]:
for name in ["impact_channels.png", "impact_recovery.png", "sqrt_impact.png"]:
    display(Image(filename=str(paths["figures"] / name)))

## Almgren–Chriss の導出・感応度・効率的フロンティア

区間 $[0,T]$ で $X$ 株を売る、連続時間の平均–分散プログラムです。線形の一時的インパクト $\eta$ と価格ボラティリティ $\sigma$ のもとで、次を最小化します。

$$J[x]=\int_0^T\!\left(\eta\,\dot{x}_t^2+\lambda\,\sigma^2 x_t^2\right)dt,\qquad x_0=X,\ x_T=0.$$

第 1 項は速く売ること（$\dot{x}$ は取引速度）のインパクトコスト、第 2 項は在庫 $x$ を保持したまま価格が拡散するタイミングリスクで、リスク回避度 $\lambda$ で重み付けされます。

Euler–Lagrange 条件は $\ddot{x}=\kappa^2 x$（緊急度 $\kappa=\sqrt{\lambda\sigma^2/\eta}$）を与え、境界条件を満たす解は次です。

$$x_t^{*}=X\,\frac{\sinh\!\left(\kappa(T-t)\right)}{\sinh(\kappa T)},\qquad x_t^{*}\to X\!\left(1-\frac{t}{T}\right)\ (\kappa\to0).$$

$\kappa\to0$ の極限が TWAP です。$\lambda$ や $\sigma$ が大きい（緊急度が高い）ほど執行は前倒しに、$\eta$ が大きい（インパクトが高価）ほど TWAP 方向へ平坦化します。注文サイズ $X$ は数量をスケールしますが、正規化した形は変えません。

決定グリッド上の子注文は $q_k=x_k-x_{k+1}$。期待ショートフォールとタイミングリスク分散は次です。

$$E[C]=\frac{\gamma}{2}X^2+\eta\sum_k\frac{q_k^2}{\Delta t},\qquad V[C]=\sum_k \sigma_k^2\,\Delta t\,x_{k+1}^2$$

（期待コストにはハーフスプレッド$\,\cdot X$ と手数料も加わります）。$\lambda$ を掃引すると、期待コストとタイミングリスク標準偏差の**効率的フロンティア**——最適な速度／リスクのトレードオフの一覧——が描けます。

In [ ]:
display(Image(filename=str(paths["figures"] / "ac_trajectories.png")))
display(Image(filename=str(paths["figures"] / "efficient_frontier.png")))

## Obizhaeva–Wang 型の回復力のある流動性

AC が速度を一時的インパクト率で罰するのに対し、OW は**回復力のある**板をモデル化します。取引は純粋に過渡的な変位 $D$ を押し上げ、率 $\rho$ で減衰します。

$$dD_t=-\rho\,D_t\,dt+\eta_t\,v_t\,dt,\qquad D_{k+1}=e^{-\rho\,\Delta t}\left(D_k+\eta_t q_k\right).$$

ブロック形状の板に対して執行すると、スケジュールの期待コストは二次形式になります。

$$C(q)=\eta_t\!\left[\tfrac{1}{2}\sum_k q_k^2+\sum_{j<k} q_k q_j\,G\!\left((k-j)\Delta t\right)\right]=\tfrac{1}{2}\eta_t\,q^{\top} M q,\qquad M_{ij}=G\!\left(|i-j|\,\Delta t\right),\ M_{ii}=1.$$

指数核（および完全単調な冪核）では $M$ は対称正定値なので、スケジューリング QP は良設定です。

2 つの解法が一致します。古典的なリスク中立 Obizhaeva–Wang（2013）閉形式は、$t=0$ と $t=T$ に等しいブロック $X/(\rho T+2)$ を置き、間を一定率 $\rho X/(\rho T+2)$ で取引します。離散ソルバは $\sum_k q_k=X,\ q\ge0$ のもとで $\tfrac{1}{2}\eta_t\,q^{\top}Mq$ を最小化し、次で解きます。

$$q=\frac{M^{-1}\mathbf{1}}{\mathbf{1}^{\top} M^{-1}\mathbf{1}}\,X$$

（小さな active-set ループ、任意の核で有効）。回復力が低い（$\rho$ が小さい）と変位が持続し忍耐が有利に、$\rho$ が高いと流動性が補充され再利用が速くできます。

本ラボはリスク中立の連続解と離散凸最適化のみを実装します。OW の**リスク回避**拡張は実装していません——リスク選好は本スイートでは Almgren–Chriss 層を通じてのみ入ります（METHODOLOGY.md 参照）。

In [ ]:
resilience = pd.read_csv(paths["data"] / "resilience_sweep.csv")
display(resilience.groupby(["rho", "method"], as_index=False)[["cost", "twap_cost"]].first())

## 反応型指値注文板・注文不均衡・イベント状態

簡略化した板は、片側あたり 1 つの集約 Level-1 気配に深い流動性密度を加えた構造で、スプレッド・注文不均衡・直近フロー・ボラティリティ・自己インパクトを追跡します。注文不均衡は次です。

$$I=\frac{Q^{b}-Q^{a}}{Q^{b}+Q^{a}}\in[-1,1].$$

**不動**仲値 $S^0$ は自分の取引に反応しません。**被影響**フェア仲値はプログラム符号 $s$ を用いて $\text{mid}=S^0-s\left(\gamma\,\Sigma_{\text{mkt}}+D\right)$、すなわち古典世界と同じ恒久 $\gamma$ と過渡 $D$ の経路です（受動的な指値約定は流動性を供給するので寄与しません）。ここでの一時的インパクトは内生的で、大口の成行注文が L1 深度を消費し、深い流動性まで板を歩き、スプレッドを広げます——別途の $\eta$ は支払いません。

潜在的な短期アルファは AR(1) $\alpha_k=\phi\,\alpha_{k-1}+\sigma_\alpha Z_k$ に従い、注文フローと次の仲値変動の両方を駆動します——これが定型化された逆選択（adverse selection）の経路です。サブステップあたりの外生成行出来高は Poisson 個数×平均サイズで、強度は cap 付きの log-linear です。

$$\lambda=\lambda_{\text{base}}\,\exp\!\left(\min(\text{tilt},\,10)\right),\qquad \text{tilt}=b_0+b_1 I+b_2 r+b_3\,\text{stress}+b_4\,\alpha.$$

事前抽選した一様乱数が戦略間の共通乱数を与えるため差は対で測られ、状態依存は強度だけに入ります。

対照（リプレイ、`reactive=False`）は同じ外生抽選を再利用しつつ、自己の痕跡を完全に取り除きます——深度は消費されず、気配と仲値は動かず、インパクト状態も不変です。反応型とリプレイの差が、自己の足跡だけを切り出します。これは透明な反応型サンドボックスであり、ヒストリカル・リプレイでも取引所級のリアリズムでもありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "lob_depth.png")))
display(Image(filename=str(paths["figures"] / "queue_imbalance.png")))

## 成行の板歩き・指値キュー・約定確率・逆選択
成行注文は即時性を買いますが、スプレッドと板歩きのコストを払います。受動的な指値はスプレッドを取り込めますが、前に並ぶキューが約定確率を下げ、期限時の後始末（cleanup）が支配的になることもあります。潜在的な短期アルファが注文フローと次の価格変動を同時に傾け、定型化された逆選択を生みます。

In [ ]:
display(Image(filename=str(paths["figures"] / "market_vs_limit.png")))
lob_summary = pd.read_csv(paths["metrics"] / "lob_strategy_summary.csv")
display(
    lob_summary[["strategy_id", "is_mean_bps", "cvar95_bps", "fill_rate", "cleanup_qty"]].round(3)
)

## 静的スケジュール・実装ショートフォール分布・TCA 分解・テールリスク
即時・TWAP・VWAP 型・POV・Almgren–Chriss・回復力考慮の各スケジュールは、同一の確率パスを共有します。実装ショートフォールが正なら、売り・買いのどちらでも不利です。潜在的なタイミング・スプレッド・一時的・恒久的・過渡的・手数料・後始末の各成分は、この合成モデル内でのみ厳密です。

In [ ]:
for name in ["strategy_schedules.png", "is_distributions.png", "tca_decomposition.png"]:
    display(Image(filename=str(paths["figures"] / name)))
classical = pd.read_csv(paths["metrics"] / "classical_strategy_summary.csv")
display(
    classical[
        ["strategy_id", "is_mean_bps", "is_mean_ci_lo_bps", "is_mean_ci_hi_bps", "cvar95_bps"]
    ].round(3)
)

## 反応型と非反応型のシミュレーション
対照ランは同じ外生抽選を再利用しつつ、自己の足跡を取り除きます。生じるコスト差は、設定した注文サイズについてこの簡易リプレイが過小評価する分を定量化します——実市場の推定値ではありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "reactive_vs_replay.png")))
display(
    pd.read_csv(paths["metrics"] / "reactive_comparison.csv")[
        ["mode", "is_mean_bps", "cvar95_bps"]
    ].round(3)
)

## RL の状態・行動・報酬・安全層・学習曲線

1 エピソードは 1 つの親注文で、エージェントは $N$ 回の意思決定を行い、その合間に板が進行します。12 個の正規化観測は、時刻・在庫・スプレッド・両側深度・不均衡・直近リターンとフロー・ボラティリティ・過渡的インパクト・出来高・未約定指値数量です。離散行動空間は $15=5\times3$：上限付きの成行倍率 $m\in\{0,0.5,1,1.5,2\}$ × ベースライン枚数 を、指値指令 $l\in\{\text{none},\text{join},\text{improve}\}$ と掛け合わせます。

**残差 RL** は Almgren–Chriss スケジュールをベースラインに使うため、$m=1$ は古典方策を厳密に再現し、ネットワークはその周りの上限付き逸脱を学びます。**自由 RL** は TWAP 相当のベースラインを使います。小さな共有ボディの actor–critic MLP を PPO のクリップ代理目的で学習します。

$$L=\mathbb{E}\!\left[\min\!\left(\varrho_t\,\hat{A}_t,\ \operatorname{clip}(\varrho_t,1-\epsilon,1+\epsilon)\,\hat{A}_t\right)\right],\qquad \varrho_t=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{\text{old}}}(a_t\mid s_t)},$$

アドバンテージは GAE $\hat{A}_t=\sum_{l\ge0}(\gamma\lambda)^l\,\delta_{t+l}$、$\delta_t=R_t+\gamma V(s_{t+1})-V(s_t)$ で与えます。学習報酬 $R_t$ は**整形**（増分コスト＋在庫・インパクト罰則）で、検証と報告するすべての比較は互いに素なホールドアウトのシードでの**経済的**実装ショートフォールを使います——整形報酬を損益として提示することはありません。

硬い安全層は、スクリプト方策でも学習方策でも一貫して支配的です：過剰執行を禁止し、子注文サイズと参加率を実現出来高の EMA に対して上限付けし、成行に価格カラーを適用し、期限で強制清算します（上限は免除だが板を歩くため懲罰的）。違反は計数・罰則化されるため、制約を「ごまかす」ことは決して報われません。

In [ ]:
display(Image(filename=str(paths["figures"] / "rl_training_history.png")))
history = pd.read_csv(paths["metrics"] / "rl_training_history.csv")
display(
    history.groupby("run_id", as_index=False)
    .tail(1)[["run_id", "episodes", "train_is_ma_bps", "best_val_is_bps"]]
    .round(3)
)

## 分布内比較とストレステスト
TWAP・Almgren–Chriss・混合ヒューリスティック・残差 RL・自由 RL を、固定したホールドアウトのシードで評価します。ストレス・レジームはボラティリティ・流動性・出来高プロファイル・不利アルファ・スプレッド／回復力を変化させます。quick プロファイルは RL シードが 1 つだけなので、シードに頑健な優位性は確立できません。

In [ ]:
display(Image(filename=str(paths["figures"] / "stress_test_heatmap.png")))
stress = pd.read_csv(paths["metrics"] / "stress_summary.csv")
display(stress.pivot(index="strategy_id", columns="regime", values="is_mean_bps").round(3))

## 特徴量アブレーション・RL 行動診断・分布シフト
各残差方策は、観測グループを 1 つずつマスクしたうえで再学習します。モデル誤指定テストは、学習済み方策を固定したまま回復力・深度・スプレッドストレス・流動性を変えます。差はシミュレータ上の証拠であって、実市場での因果的な特徴量重要度ではありません。

In [ ]:
display(Image(filename=str(paths["figures"] / "feature_ablation.png")))
display(Image(filename=str(paths["figures"] / "model_misspecification.png")))
display(
    pd.read_csv(paths["metrics"] / "ablation_summary.csv")[
        ["strategy_id", "feature_removed", "is_mean_bps", "delta_vs_full_bps"]
    ].round(3)
)

## この実験が示すこと
記録済みシードのもとで実装が再現可能であること、古典的な緊急度がリスクとインパクトのパラメータに正しく反応すること、エージェントが自分の合成市場を変えること、指値注文がスプレッドと約定リスクを交換すること、方策の順位がストレスとモデルシフトのもとで変わり得ること——これらを示します。

In [ ]:
best_classical = classical.loc[classical.is_mean_bps.idxmin()]
best_lob = lob_summary.loc[lob_summary.is_mean_bps.idxmin()]
id_test = stress[stress.regime == "in_distribution"]
best_id = id_test.loc[id_test.is_mean_bps.idxmin()]
display(
    Markdown(f"""**Generated quick-run findings**  
- Lowest classical mean cost: `{best_classical.strategy_id}` at **{best_classical.is_mean_bps:.3f} bps**.  
- Lowest reactive-LOB mean cost: `{best_lob.strategy_id}` at **{best_lob.is_mean_bps:.3f} bps**.  
- Lowest in-distribution comparison cost: `{best_id.strategy_id}` at **{best_id.is_mean_bps:.3f} bps**.  
These rankings are conditional on the synthetic calibration.""")
)

## この実験が示さないこと・限界と次の一歩
本ラボは、実市場での収益性・因果的な特徴量価値・取引所級のリアリズム・統計的な RL 優位性を示しません。隠れ流動性・レイテンシ・複数会場ルーティング・戦略的な対抗者・相場操縦制約・クロスインパクト・経験的キャリブレーションを省いています。次の一歩は、厳格なホールドアウトでの実データ較正、マルチアセットのクロスインパクト、ディーラー／RFQ 市場、確率的な回復力、より安全な制約付き RL、分布的にロバストな評価、そしてラフボラティリティや Hawkes フローとの相互作用です。

## 参考文献

Almgren–Chriss、Obizhaeva–Wang (2013)、PPO はコード内で直接参照しています。それ以外は、実装したアイディアおよび「限界と次の一歩」のロードマップ項目に対する正典的な出典です。

**コア戦略モデル**
- R. Almgren & N. Chriss (2001), "Optimal Execution of Portfolio Transactions," *Journal of Risk* 3(2) — 平均–分散の執行プログラム、閉形式の `sinh` 軌道、効率的フロンティア（`almgren_chriss.py` で実装）。
- D. Bertsimas & A. Lo (1998), "Optimal Control of Execution Costs," *Journal of Financial Markets* 1(1) — Almgren–Chriss の前身となるリスク中立の動的計画法。
- A. Obizhaeva & J. Wang (2013), "Optimal Trading Strategy and Supply/Demand Dynamics," *Journal of Financial Markets* 16(1) — 回復力のある需給と「端点ブロック＋一定率＋端点ブロック」スケジュール（`resilience.py` で実装）。

**インパクトと市場マイクロストラクチャーの基盤**
- J.-P. Bouchaud, Y. Gefen, M. Potters & M. Wyart (2004), "Fluctuations and Response in Financial Markets," *Quantitative Finance* 4(2) — 過渡的経路を一般化する伝播核（減衰核）の視点。
- J. Gatheral (2010), "No-Dynamic-Arbitrage and Market Impact," *Quantitative Finance* 10(7) — インパクトと減衰核の間の無裁定整合条件（ロードマップ項目）。
- R. Almgren, C. Thum, E. Hauptmann & H. Li (2005), "Direct Estimation of Equity Market Impact," *Risk* 18(7) — `sqrt_impact` 診断の背後にある平方根則の実証推定。
- B. Tóth ほか (2011), "Anomalous Price Impact and the Critical Nature of Liquidity in Financial Markets," *Physical Review X* 1(2) — 平方根則の普遍性。
- A. Kyle (1985), "Continuous Auctions and Insider Trading," *Econometrica* 53(6) — 線形の恒久インパクトと、潜在アルファによる逆選択経路の背後にある情報フローの視点。
- A. Perold (1988), "The Implementation Shortfall: Paper Versus Real Portfolios," *Journal of Portfolio Management* 14(3) — 本ラボ全体で用いるコスト指標。

**強化学習**
- J. Schulman, F. Wolski, P. Dhariwal, A. Radford & O. Klimov (2017), "Proximal Policy Optimization Algorithms," arXiv:1707.06347 — `rl_training.py` の PPO クリップ代理目的。
- J. Schulman, P. Moritz, S. Levine, M. Jordan & P. Abbeel (2016), "High-Dimensional Continuous Control Using Generalized Advantage Estimation," ICLR / arXiv:1506.02438 — GAE。
- Y. Nevmyvaka, Y. Feng & M. Kearns (2006), "Reinforcement Learning for Optimized Trade Execution," *ICML* — 執行×強化学習の嚆矢。
- D. Hendricks & D. Wilcox (2014), "A Reinforcement Learning Extension to the Almgren–Chriss Framework for Optimal Trade Execution," *IEEE CIFEr* — AC ベースライン周りの RL 調整、残差 RL の発想。

**シミュレータ設計と次の一歩**
- W. Huang, C.-A. Lehalle & M. Rosenbaum (2015), "Simulating and Analyzing Order Book Data: The Queue-Reactive Model," *Journal of the American Statistical Association* 110(509) — 反応型板の正典。
- J. Gatheral, T. Jaisson & M. Rosenbaum (2018), "Volatility is Rough," *Quantitative Finance* 18(6) — ラフボラティリティ・レジーム（ロードマップ項目）。
- E. Bacry, I. Mastromatteo & J.-F. Muzy (2015), "Hawkes Processes in Finance," *Market Microstructure and Liquidity* 1(1) — Hawkes 型の注文フロー（ロードマップ項目）。

**教科書**
- Á. Cartea, S. Jaimungal & J. Penalva (2015), *Algorithmic and High-Frequency Trading*, Cambridge University Press.
- O. Guéant (2016), *The Financial Mathematics of Market Liquidity: From Optimal Execution to Market Making*, CRC Press.